# 🌐 Wikidata SPARQL Exploration — EDM Taxonomy & Genre Hierarchy

This notebook queries the Wikidata SPARQL endpoint (`https://query.wikidata.org/sparql`) to evaluate graph-based parent-child genre relationships and ontological metadata for electronic music (EDM) sub-genre classification.

### Objectives
* Test endpoint access using `fetch_raw_api_sample` with required `User-Agent` identification.
* Query sub-genres under Electronic Dance Music (`wd:Q212805`) using SPARQL `wdt:P279*` (subclass of).
* Transform raw SPARQL JSON bindings into structured tabular DataFrames using `transform_sparql_bindings_to_dataframe`.
* Test entity relationship mapping (`wdt:P136` = genre) to cross-reference artists with the EDM sub-genre taxonomy.

In [1]:
import pandas as pd
from src.agies.integration.api_helpers import (
    fetch_raw_api_sample, 
    transform_sparql_bindings_to_dataframe)

In [2]:
# Global Endpoint & Header Configuration
WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"
HEADERS = {
    "User-Agent": "AGIES/0.1 (info@dataravers.space)",
    "Accept": "application/sparql-results+json"
}

## 1. Fetch & Parse EDM Genre Taxonomy

Query sub-genres under Electronic Dance Music (`wd:Q212805`) along with their optional parent genre relationships (`wdt:P279`), and transform the raw JSON response directly into a Pandas DataFrame.

In [3]:
# SPARQL Query: Fetch sub-genres under EDM (Q212805) and optional parent genre mappings
sparql_taxonomy = """
SELECT ?genre ?genreLabel ?parentGenre ?parentGenreLabel WHERE {
  ?genre wdt:P279* wd:Q212805 .
  OPTIONAL { ?genre wdt:P279 ?parentGenre . }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
LIMIT 20
"""

# Fetch raw JSON response
raw_taxonomy_data = fetch_raw_api_sample(
    url=WIKIDATA_SPARQL_URL,
    params={"query": sparql_taxonomy, "format": "json"},
    headers=HEADERS,
    timeout=15
)

# Transform SPARQL bindings to flat DataFrame
df_taxonomy = transform_sparql_bindings_to_dataframe(raw_taxonomy_data)
df_taxonomy.head(10)

,genre,parentGenre,genreLabel,parentGenreLabel
0,http://www.wikidata.org/entity/Q212805,http://www.wikidata.org/entity/Q7075,digital library,library
1,http://www.wikidata.org/entity/Q212805,http://www.wikidata.org/entity/Q7094076,digital library,online database
2,http://www.wikidata.org/entity/Q1065413,http://www.wikidata.org/entity/Q8513,institutional repository,database
3,http://www.wikidata.org/entity/Q1065413,http://www.wikidata.org/entity/Q1235234,institutional repository,document repository
4,http://www.wikidata.org/entity/Q1065413,http://www.wikidata.org/entity/Q3133368,institutional repository,source code repository
5,http://www.wikidata.org/entity/Q1224870,http://www.wikidata.org/entity/Q186165,virtual library,web portal
6,http://www.wikidata.org/entity/Q1224870,http://www.wikidata.org/entity/Q212805,virtual library,digital library
7,http://www.wikidata.org/entity/Q1224984,http://www.wikidata.org/entity/Q166118,digital archive,archives
8,http://www.wikidata.org/entity/Q1224984,http://www.wikidata.org/entity/Q212805,digital archive,digital library
9,http://www.wikidata.org/entity/Q1224984,http://www.wikidata.org/entity/Q140500284,digital archive,archive


## 2. Query Sample EDM Artists by Sub-Genre

Test the genre attribute relationship (`wdt:P136`) by fetching sample artists tagged under Electronic Dance Music sub-genres to verify cross-referencing capabilities.

In [4]:
# SPARQL Query: Fetch sample artists associated with EDM sub-genres (wdt:P136 = genre)
sparql_artists = """
SELECT ?artist ?artistLabel ?genreLabel WHERE {
  ?artist wdt:P136 ?genre .
  ?genre wdt:P279* wd:Q212805 .
  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
LIMIT 20
"""

# Fetch raw JSON response
raw_artist_data = fetch_raw_api_sample(
    url=WIKIDATA_SPARQL_URL,
    params={"query": sparql_artists, "format": "json"},
    headers=HEADERS,
    timeout=15
)

# Transform SPARQL bindings to flat DataFrame
df_artists = transform_sparql_bindings_to_dataframe(raw_artist_data)
df_artists.head(10)

,artist,artistLabel,genreLabel
0,http://www.wikidata.org/entity/Q983071,Sergio Martino,digital library
